# AI Livestream Commerce VN — Colab vLLM Demo

Canonical backend `/api/v1` mock-render demo. No local dataset or checkpoint is required.


In [ ]:
# Preflight Check: fast, fail-loud, and network-free.
from pathlib import Path
import importlib
import os
import sys

REPO_DIR = Path("/content/ai-livestream-commerce-vn")
OUTPUT_DIR = Path("/content/ai-livestream-demo")
REPO_URL = "https://github.com/justHman/ai-livestream-commerce-vn.git"
REPO_BRANCH = "main"
VLLM_MODEL_ID = "cyankiwi/Qwen3.5-4B-AWQ-4bit"
USE_CLOUD_LIVEAVATAR = False

errors = []
for module_name in ("torch", "IPython", "google.colab"):
    try:
        importlib.import_module(module_name)
    except Exception as exc:
        errors.append(f"cannot import {module_name}: {exc}")
if errors:
    raise RuntimeError("Preflight failed:\n- " + "\n- ".join(errors))

from google.colab import userdata
import torch

if "google.colab" not in sys.modules:
    raise RuntimeError("Preflight failed: this notebook must run inside Google Colab")


def load_secret(name: str) -> str:
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if value:
        os.environ[name] = value
    return value or ""


LIVEAVATAR_API_KEY = load_secret("LIVEAVATAR_API_KEY")
NGROK_AUTHTOKEN = load_secret("NGROK_AUTHTOKEN")
tried_paths = [Path("/content"), OUTPUT_DIR.parent, OUTPUT_DIR]
errors = []
if not torch.cuda.is_available():
    errors.append("CUDA GPU is required because this notebook runs vLLM")
if not NGROK_AUTHTOKEN:
    errors.append("NGROK_AUTHTOKEN is required because the notebook opens an ngrok tunnel")
if USE_CLOUD_LIVEAVATAR and not LIVEAVATAR_API_KEY:
    errors.append("LIVEAVATAR_API_KEY is required when USE_CLOUD_LIVEAVATAR=True")
if not REPO_URL.startswith("https://") or not REPO_BRANCH or not VLLM_MODEL_ID:
    errors.append("REPO_URL, REPO_BRANCH, and VLLM_MODEL_ID must be nonempty valid configuration")
try:
    OUTPUT_DIR.parent.mkdir(parents=True, exist_ok=True)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    probe = OUTPUT_DIR / ".preflight-write-probe"
    probe.write_text("ok", encoding="utf-8")
    probe.unlink()
except OSError as exc:
    errors.append(f"output path is not writable: {exc}")
if errors:
    raise RuntimeError(
        "Preflight failed:\n- " + "\n- ".join(errors)
        + "\nTried paths:\n- " + "\n- ".join(map(str, tried_paths))
    )

print(f"Environment: Colab GPU={torch.cuda.get_device_name(0)}")
print(f"Output: {OUTPUT_DIR}")
print(f"Planned repository path after clone: {REPO_DIR}")
print(f"Repository: {REPO_URL} ({REPO_BRANCH})")
print(f"vLLM model source: {VLLM_MODEL_ID}")
print("No local dataset, tensor, or checkpoint input is required by this demo.")


In [ ]:
# Clone the repository after preflight validates the configuration.
import subprocess

if not (REPO_DIR / ".git").exists():
    subprocess.check_call(["git", "clone", "--branch", REPO_BRANCH, "--depth", "1", REPO_URL, str(REPO_DIR)])
else:
    subprocess.check_call(["git", "fetch", "origin", REPO_BRANCH, "--depth", "1"], cwd=REPO_DIR)
    subprocess.check_call(["git", "checkout", REPO_BRANCH], cwd=REPO_DIR)
    subprocess.check_call(["git", "reset", "--hard", f"origin/{REPO_BRANCH}"], cwd=REPO_DIR)
print(f"Repository ready: {REPO_DIR}")


In [ ]:
# Install the editable project and vLLM in the Colab runtime.
import subprocess
import sys

BACKEND_SRC = REPO_DIR / "services/product/backend_service/src"
sys.path[:0] = [str(BACKEND_SRC), str(REPO_DIR)]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR / "services/product/backend_service"), "vllm", "pyngrok"])
from providers.liveavatar_cloud.service import colab_server
print("Provider module import passed")


In [ ]:
# Launch backend.main with the canonical backend /api/v1 contract.
import os
import subprocess
import sys

os.environ.update({
    "PYTHONPATH": f"{REPO_DIR / 'services/product/backend_service/src'}:{REPO_DIR}:{os.environ.get('PYTHONPATH', '')}",
    "APP_ENV": "dev",
    "RENDER_BACKEND": "mock",
    "SESSION_STORE": "memory",
    "LLM_ENGINE": "vllm",
    "LLM_MODEL": VLLM_MODEL_ID,
    "TTS_ENGINE": "tone",
    "DIRECTOR_ENABLED": "0",
    "BACKEND_API_TOKEN": "",
    "ADMIN_API_TOKEN": "",
})
if USE_CLOUD_LIVEAVATAR:
    os.environ["LIVEAVATAR_API_KEY"] = LIVEAVATAR_API_KEY
LOG_PATH = OUTPUT_DIR / "backend_uvicorn.log"
log_file = LOG_PATH.open("w", encoding="utf-8")
SERVER_PROC = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "backend.main:create_app", "--factory", "--host", "0.0.0.0", "--port", "8000"],
    cwd=REPO_DIR,
    env=os.environ.copy(),
    stdout=log_file,
    stderr=subprocess.STDOUT,
    text=True,
)
print(f"Canonical backend PID: {SERVER_PROC.pid}; log: {LOG_PATH}")


In [ ]:
# Expose the canonical backend demo; preflight already required the tunnel token.
from pyngrok import ngrok

ngrok.set_auth_token(NGROK_AUTHTOKEN)
tunnel = ngrok.connect(8000, bind_tls=True)
PUBLIC_URL = tunnel.public_url.rstrip("/")
print(f"Frontend backend origin: {PUBLIC_URL}")
print(f"Canonical backend API base: {PUBLIC_URL}/api/v1")


In [ ]:
# Start and stop a mock canonical backend /api/v1/lite session, then write an artifact.
import json
import time
import httpx

base_url = "http://127.0.0.1:8000/api/v1"
deadline = time.time() + 90
while True:
    try:
        ready_response = httpx.get(f"{base_url}/health/ready", timeout=10)
        if ready_response.status_code == 200:
            break
    except httpx.HTTPError:
        pass
    if time.time() >= deadline:
        raise RuntimeError(f"Canonical backend readiness failed; inspect {LOG_PATH}")
    time.sleep(2)

artifact = OUTPUT_DIR / "mock-session-smoke.json"
try:
    sessions_response = httpx.get(f"{base_url}/sessions", timeout=30)
    sessions_response.raise_for_status()
    sessions = sessions_response.json()
finally:
    artifact.write_text(
        json.dumps(
            {"ready": ready_response.json(), "sessions": sessions},
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )
size_mb = artifact.stat().st_size / (1024 * 1024)
print(f"Loaded API response: shape=n/a dtype=json size={size_mb:.3f} MB source={base_url}/sessions")
print(f"Session listing fetched: {sessions}")
print(f"Saved: {artifact}")


## Frontend

Paste the printed **Frontend backend origin** into `frontend/lite.html`. The
frontend appends `/api/v1` itself, so do not paste a URL already ending in
`/api/v1`. The standalone provider server remains on `/api`.


In [ ]:
# Final report and compact recursive output tree.
from pathlib import Path


def print_output_tree(root_path: Path) -> None:
    print(root_path)
    for directory, subdirs, files in os.walk(root_path):
        current = Path(directory)
        depth = len(current.relative_to(root_path).parts)
        indent = "  " * depth
        for name in sorted(subdirs)[:2]:
            print(f"{indent}{name}/")
        if len(subdirs) > 2:
            print(f"{indent}.../")
        for name in sorted(files)[:2]:
            print(f"{indent}{name}")
        if len(files) > 2:
            print(f"{indent}...")


print_output_tree(OUTPUT_DIR)
print(f"Frontend backend origin: {globals().get('PUBLIC_URL', 'local')}")
print(f"Canonical backend API base: {globals().get('PUBLIC_URL', 'local')}/api/v1")
print("Paste the backend origin into frontend/lite.html; it appends /api/v1. Standalone provider demos use /api.")


def shutdown_demo() -> None:
    try:
        if SERVER_PROC.poll() is None:
            SERVER_PROC.terminate()
            SERVER_PROC.wait(timeout=15)
    finally:
        if not log_file.closed:
            log_file.close()
        ngrok.kill()
    print("Demo stopped")
